# 1. Merge data

In [13]:
import pandas as pd
import re

keys = ["REF_AREA"]
drop_cols = []

country_continent = pd.read_csv("../data/countryContinent.csv", encoding='latin-1')
pm25 = pd.read_csv("../data/pm25.csv", encoding='latin-1')
gdp = pd.read_csv("../data/gdp.csv", encoding='latin-1')
population = pd.read_csv("../data/population.csv", encoding='latin-1')

pm25_years = set()
for c in pm25.columns:
    m = re.search(r"(\d{4})", c)
    if m:
        pm25_years.add(m.group(1))

print(f"Years in PM2.5 data: {sorted(pm25_years)}")

def prefix_year_columns(df, prefix, keep_keys=keys, filter_years=None):
    df = df.copy()
    rename = {}
    year_new_names = []
    for c in df.columns:
        m = re.search(r"(\d{4})", c)
        if m:
            year = m.group(1)
            if filter_years is None or year in filter_years:
                rename[c] = f"{prefix}_{year}"
                year_new_names.append(rename[c])
    df = df.rename(columns=rename)
    cols = [k for k in keep_keys if k in df.columns] + [c for c in year_new_names if c in df.columns]
    return df[cols]

pm25_pref = prefix_year_columns(pm25, "pm25", keep_keys=["REF_AREA", "REF_AREA_LABEL"])
gdp_pref = prefix_year_columns(gdp, "gdp", filter_years=pm25_years)
pop_pref = prefix_year_columns(population, "pop", filter_years=pm25_years)

pm25_enriched = pm25_pref.merge(gdp_pref, on=keys, how="left").merge(pop_pref, on=keys, how="left")

continent_mapping = country_continent[["code_3", "continent"]].rename(columns={"code_3": "REF_AREA"})
pm25_enriched = pm25_enriched.merge(continent_mapping, on=keys, how="left")

cols = pm25_enriched.columns.tolist()
cols.remove('continent')
ref_area_index = cols.index('REF_AREA')
cols.insert(ref_area_index + 1, 'continent')
pm25_enriched = pm25_enriched[cols]

to_drop = [c for c in drop_cols if c in pm25_enriched.columns]
if to_drop:
    pm25_enriched = pm25_enriched.drop(columns=to_drop)

Years in PM2.5 data: ['1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020']


# 2. Clean data

In [14]:
id_vars = ['REF_AREA', 'continent', 'REF_AREA_LABEL']
value_vars = [c for c in pm25_enriched.columns if c not in id_vars]

pm25_long = pm25_enriched.melt(id_vars=id_vars, value_vars=value_vars, var_name='factor_year', value_name='value')

pm25_long[['factor', 'year']] = pm25_long['factor_year'].str.split('_', n=1, expand=True)
pm25_long = pm25_long.drop(columns=['factor_year'])

pm25_long['year'] = pm25_long['year'].astype(int)

pm25_long = pm25_long[['REF_AREA', 'continent', 'REF_AREA_LABEL', 'year', 'factor', 'value']]

pm25_long = pm25_long.sort_values(['REF_AREA', 'year', 'factor']).reset_index(drop=True)

print(f"Shape: {pm25_long.shape}")
print(pm25_long.head(5))

pm25_long.to_csv("../data/processed_data.csv", index=False)

Shape: (18414, 6)
  REF_AREA continent REF_AREA_LABEL  year factor         value
0      AFG      Asia    Afghanistan  1990    gdp           NaN
1      AFG      Asia    Afghanistan  1990   pm25  6.417410e+01
2      AFG      Asia    Afghanistan  1990    pop  1.204566e+07
3      AFG      Asia    Afghanistan  1991    gdp           NaN
4      AFG      Asia    Afghanistan  1991   pm25  6.418815e+01
